# Keyword Trends: Topic Emergence in Author Keywords

This notebook tracks what biomedical research talks about through author-supplied keywords: which topics rose, which faded, and when new subjects emerged. Unlike MeSH (a controlled vocabulary), author keywords are free text, so the first task is cleaning and normalizing them; only then are trends meaningful.

Two facts from the EDA shape the whole notebook. Keywords are near-absent before about 2012 and cross 50% coverage only around 2015, so the timeline starts at 2015; earlier years are too sparse to trend. And early keyword records contain non-biomedical noise (stray metadata, table labels, and similar), so a noise filter is applied. All trends are normalized per 1,000 articles so a rising keyword reflects real adoption rather than corpus growth.

Runs on the published metadata (keywords are included; no abstracts needed).

## Setup

In [ ]:
import os, glob, collections, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING: pick ONE option (same pattern as 03_eda / 04 / 05)
# =====================================================================

# ---- OPTION A: LOCAL (active) ----
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")

# ---- OPTION B: KAGGLE (uncomment on Kaggle) ----
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input; attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])
# =====================================================================

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "keywords", "n_keywords"])
df = df[(df["year"] >= 2015) & (df["year"] <= 2025)].copy()   # keywords reliable from ~2015 (EDA 7a)
print(f"loaded {len(df):,} records (2015-2025)")
assert len(df) == df["uid"].nunique(), "duplicate PMIDs; dedup did not run"
print(f"articles with at least one keyword: {(df['n_keywords'] > 0).mean()*100:.0f}%")
df[["year", "n_keywords"]].head(3)

## 1. Normalize the keywords

Author keywords arrive in many surface forms: different casing ("COVID-19", "covid-19"), punctuation ("COVID 19" vs "COVID-19"), and spacing. Without normalization the same concept splits across many variants and no trend is reliable. The function below lowercases, standardises punctuation and whitespace, and drops too-short tokens. The effect is measured by how far the distinct-keyword count drops.

In [ ]:
def normalize_kw(kw):
    """Lowercase, standardise punctuation/whitespace; return None if too short to be meaningful."""
    if not isinstance(kw, str):
        return None
    k = kw.strip().lower()
    k = k.replace("-", " ").replace("_", " ").replace("/", " ")
    k = re.sub(r"[^\w\s]", "", k)          # drop stray punctuation
    k = " ".join(k.split())                # collapse whitespace
    return k if len(k) >= 2 else None

def normalize_list(lst):
    if not isinstance(lst, (list, np.ndarray)):
        return []
    out = []
    for kw in lst:
        n = normalize_kw(kw)
        if n:
            out.append(n)
    return out

df["kw_norm"] = df["keywords"].map(normalize_list)

raw_distinct = len(set(k for lst in df["keywords"] if isinstance(lst, (list, np.ndarray))
                       for k in lst if isinstance(k, str)))
norm_distinct = len(set(k for lst in df["kw_norm"] for k in lst))
print(f"distinct keywords, raw:        {raw_distinct:,}")
print(f"distinct keywords, normalized: {norm_distinct:,}")
print(f"reduction: {(1 - norm_distinct/raw_distinct)*100:.0f}% (casing/punctuation variants merged)")

**What this shows:** normalization merges surface variants of the same keyword, cutting the distinct-keyword count substantially. This matters because an un-normalized trend would split a single rising topic across several spellings and understate it. The remaining vocabulary is still large (author keywords are open-ended), but each entry now corresponds to a concept rather than a spelling.

## 2. Filter non-biomedical noise

The EDA noted that early keyword records contain stray non-topical strings (metadata, table and figure labels, placeholder values). These are removed by an explicit noise list plus a rule against purely numeric tokens. The list is shown so it can be inspected and extended; removal is evidence-led, the candidates come from the most frequent non-topical terms in the data.

In [ ]:
# inspect the most frequent normalized keywords first (the evidence for what is noise)
kw_freq = collections.Counter(k for lst in df["kw_norm"] for k in lst)
print("top 30 normalized keywords by frequency (scan for non-topical noise):")
for k, c in kw_freq.most_common(30):
    print(f"  {k:<35} {c:,}")

In [ ]:
# Noise list: non-topical strings and placeholders identified from the frequency scan above.
NOISE = {
    "na", "n a", "none", "table 1", "figure 1", "fig 1", "supplementary",
    "introduction", "methods", "results", "conclusion", "conclusions",
    "background", "objective", "objectives", "discussion",
    "ames research center", "nasa", "research article", "original article",
}

def is_noise(k):
    if k in NOISE:
        return True
    if k.isdigit():                 # purely numeric tokens are not topics
        return True
    return False

def clean_list(lst):
    return [k for k in lst if not is_noise(k)]

df["kw_clean"] = df["kw_norm"].map(clean_list)
clean_distinct = len(set(k for lst in df["kw_clean"] for k in lst))
print(f"distinct keywords after noise filter: {clean_distinct:,}")
print(f"removed {norm_distinct - clean_distinct:,} noise/placeholder terms")

**What this shows:** the noise filter removes placeholder and structural strings that are not research topics. The reduction is modest in vocabulary terms but improves the trends: without it, terms like section headers would appear among the "top keywords" and clutter any topic analysis. The cleaned list is the basis for all trends below.

## 3. Most common keywords

The overall most frequent cleaned keywords across 2015-2025. This is the topical profile of the recent corpus as authors describe it themselves (distinct from MeSH, which is assigned by indexers).

In [ ]:
kw_freq = collections.Counter(k for lst in df["kw_clean"] for k in lst)
top = pd.Series(dict(kw_freq.most_common(20)))[::-1]
plt.figure(figsize=(10, 8))
sns.barplot(x=top.values, y=top.index, color="#1d6fb8")
plt.title("Top 20 author keywords (2015-2025, cleaned)")
plt.xlabel("articles"); plt.ylabel("")
plt.tight_layout(); plt.show()

**What this shows:** the most common author keywords reflect the dominant research themes of the period as authors frame them. Compared with MeSH (notebook 04), author keywords tend to be more specific and current (naming diseases, methods, and emerging topics directly), which is what makes them useful for tracking emergence.

## 4. Keyword trends over time (normalized per 1,000 articles)

Raw keyword counts rise with corpus volume, so they are normalized per 1,000 articles per year. This shows genuine changes in prevalence. A helper computes the per-1,000 series for any keyword; the most frequent terms are plotted.

In [ ]:
articles_per_year = df.groupby("year").size()

def keyword_per_1000(term):
    has = df["kw_clean"].map(lambda l: term in l)
    return (df.assign(h=has).groupby("year")["h"].sum() / articles_per_year * 1000).reindex(articles_per_year.index)

top_terms = [k for k, _ in kw_freq.most_common(10)]
plt.figure(figsize=(13, 6))
for term in top_terms:
    s = keyword_per_1000(term)
    plt.plot(s.index, s.values, marker="o", markersize=3, label=term)
plt.title("Top keyword prevalence over time (per 1,000 articles)")
plt.xlabel("year"); plt.ylabel("per 1,000 articles")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

**What this shows:** normalized prevalence separates genuine topic growth from corpus growth. A keyword whose per-1,000 rate climbs is being adopted more widely, not just appearing more because there are more papers. Flat lines are stable topics; rising or falling lines are shifting research attention.

## 5. Emerging keywords

The clearest signal is what is common now but was absent before: keywords with high prevalence in the most recent years (2022-2025) and near-zero earlier (2015-2017). This surfaces genuinely new topics rather than persistent ones.

In [ ]:
def rate_in(lo, hi):
    sub = df[(df["year"] >= lo) & (df["year"] <= hi)]
    n = len(sub)
    cnt = collections.Counter(k for lst in sub["kw_clean"] for k in lst)
    return {k: v / n * 1000 for k, v in cnt.items()}, n

early_rate, n_early = rate_in(2015, 2017)
late_rate,  n_late  = rate_in(2022, 2025)

# emerged: meaningfully present recently, near-absent early, with a minimum volume
emerged = []
for k, r_late in late_rate.items():
    r_early = early_rate.get(k, 0)
    if r_late >= 1.0 and r_early < 0.2 and late_rate[k] * n_late / 1000 >= 50:
        emerged.append((k, r_early, r_late))
emerged.sort(key=lambda x: -x[2])

print(f"keywords that emerged in 2022-2025 (per 1,000: early -> late):\n")
for k, e, l in emerged[:20]:
    print(f"  {l:6.1f}  (was {e:4.1f})   {k}")

**What this shows:** these keywords became prevalent only recently and were essentially absent in 2015-2017. Pandemic-related terms are the clearest example, but the list also surfaces newer methods and topics. Because the comparison is on per-1,000 rates rather than raw counts, the list reflects genuine emergence, not terms that merely grew with the corpus.

## 6. Declining keywords

The mirror image: keywords that were prevalent early but faded. These mark topics that lost relative attention over the period.

In [ ]:
declined = []
for k, r_early in early_rate.items():
    r_late = late_rate.get(k, 0)
    if r_early >= 1.0 and r_late < r_early * 0.4 and early_rate[k] * n_early / 1000 >= 50:
        declined.append((k, r_early, r_late))
declined.sort(key=lambda x: -(x[1] - x[2]))

print(f"keywords that declined from 2015-2017 to 2022-2025 (per 1,000: early -> late):\n")
for k, e, l in declined[:20]:
    print(f"  {e:6.1f} -> {l:4.1f}   {k}")

**What this shows:** these topics lost relative prevalence. A decline in per-1,000 rate means the topic occupies a smaller share of research attention than it did, whether because it matured, was renamed, or was displaced by newer framings. Declines are usually gentler than emergences (topics fade rather than vanish), so the threshold here is relative (a drop to under 40% of the early rate).

## 7. Summary and caveats

### What this notebook produced
This notebook cleaned and normalized free-text author keywords, then tracked topical change across 2015-2025: the most common keywords, normalized prevalence trends, and the keywords that emerged or declined. Normalization merged surface variants, a noise filter removed non-topical strings, and all trends were computed per 1,000 articles so that adoption is separated from corpus growth.

### Caveats

**Author keywords are free text and only partially normalized.** The normalization merges casing, punctuation, and spacing variants, but it does not resolve synonyms (for example "heart attack" and "myocardial infarction" remain separate) or acronyms against their expansions. A fuller analysis would map keywords to a controlled vocabulary; this notebook treats normalized surface forms as the unit.

**The timeline starts at 2015 by necessity.** Keywords are near-absent before about 2012 and reach 50% coverage only around 2015 (EDA 7a). Earlier years are excluded because their sparse, unrepresentative keywords would distort any trend. This is a recent-period analysis, not a 30-year one.

**Coverage still rises across the window.** Even within 2015-2025, keyword coverage increases (from roughly half to over 80% of articles). The per-1,000 normalization controls for article volume but not for this rising coverage, so a keyword can appear to grow partly because more papers carry keywords at all. Emergence of genuinely new terms (like pandemic vocabulary) is robust to this, but gentle trends should be read with the coverage rise in mind.

**The noise filter is a judgement call.** The NOISE set is built from the most frequent non-topical strings observed, but it is not exhaustive; rare placeholder terms may remain, and a borderline term could be argued either way. The list is explicit in section 2 so it can be extended.

---

## Notebook complete

This notebook tracked topic emergence and decline in author keywords across 2015-2025.

**Next up:** proceed to `07_covid19_timeseries.ipynb` for a focused case study on the pandemic literature. Union "COVID-19" with "Coronavirus Infections", "SARS Virus", and "Betacoronavirus" to capture the pre-2020 baseline, and report absolute counts alongside shares because the 2020-2021 surge is inflated by rapid indexing.